In [1]:
import torch
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
import os
import numpy as np

from src.model import AntiArtifactModel
from src.preprocess import AudioPreprocessor

/Users/jacobmitani/anaconda3/envs/native_env/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

model = AntiArtifactModel(embed_dim=128).to(device)

weights_path = os.path.join('weights', 'anti_artifact_model_final.pth')

model.load_state_dict(torch.load(weights_path, map_location=device))
model.eval()

processor = AudioPreprocessor(sample_rate=44100, n_fft=2048, hop_length=512)
print("model and preprocessor loaded and ready for evaluation")

Using device: mps


FileNotFoundError: [Errno 2] No such file or directory: 'weights/anti_artifact_model_final.pth'

In [ ]:
primary_path = 'eval_data/sample_clean_target'
reference_path = 'eval_data/sample_reference.wav'

clean_primary_wav = processor.wav_to_tensors(primary_path)
reference_wav = processor.wav_to_tensors(reference_path)

min_len = min(clean_primary_wav.shape[-1], reference_wav.shape[-1])
clean_primary_wav = clean_primary_wav[:, :min_len]
reference_wav = reference_wav[:, :min_len]

#augment data
def augment_waveform_eval(primary_wav, ref_wav):
    bleed_signal = ref_wav * 0.25

    delay_samples = int(processor.sample_rate * 0.65)
    decay_factor = 0.4
    reflection = torch.zeros_like(primary_wav)
    if primary_wav.shape[-1] > delay_samples:
        reflection[:, delay_samples:] = primary_wav[:, :-delay_samples] * decay_factor

    noise_floor = torch.randn_like(primary_wav) * 0.005
    artifacted_primary = primary_wav + bleed_signal + reflection + noise_floor

    # Anti-clipping
    max_val = torch.max(torch.abs(artifacted_primary))
    if max_val > 1.0:
        artifacted_primary = artifacted_primary / max_val
    return artifacted_primary

artifacted_wav = augment_waveform_eval(clean_primary_wav, reference_wav)

print("audio loaded, cross-track bleed applied.")



In [ ]:
artifacted_wav = artifacted_wav.to(device)
reference_wav = reference_wav.to(device)

complex_artifacted = processor.waveform_to_complex_stft(artifacted_wav)
complex_reference = processor.waveform_to_complex_stft(reference_wav)

mag_artifacted = torch.abs(complex_artifacted).unsqueeze(0).unsqueeze(0)
mag_reference = torch.abs(complex_reference).unsqueeze(0).unsqueeze(0)

with torch.no_grad():
    predicted_mask = model(mag_artifacted, mag_reference)

    predicted_mag = mag_artifacted * predicted_mask

print("inference completed.")



In [ ]:
clean_mag = predicted_mag.squeeze().cpu().numpy()
original_mag = mag_artifacted.squeeze().cpu.numpy()

plt.figure(figsize=(15, 6))

#plot 1: prior to separation
plt.subplot(1, 2, 1)
librosa.display.specshow(librosa.amplitude_to_db(original_mag, ref=np.max), 
                         sr=processor.sample_rate, hop_length=processor.hop_length,
                         y_axis='hz', x_axis='time')
plt.title('Before: Artifacted Input (Target + Bleed)')
plt.colorbar(format='%+2.0f dB')

# Plot 2: After Separation
plt.subplot(1, 2, 2)
librosa.display.specshow(librosa.amplitude_to_db(clean_mag, ref=np.max), 
                         sr=processor.sample_rate, hop_length=processor.hop_length,
                         y_axis='hz', x_axis='time')
plt.title('After: Cross-Track Attention Output')
plt.colorbar(format='%+2.0f dB')

plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    clean_audio_tensor = processor.reconstruct_audio(
        complex_artifacted.unsqueeze(0),
        predicted_mask
    )

clean_audio_np = clean_audio_tensor.squeeze().cpu().numpy()

output_path = 'eval_data/separated_output.wav'
sf.write(output_path, clean_audio_np, processor.sample_rate)

print(f"audio saved to: {output_path}")